# Day 2: measurements, uncertainty and decisions

Work through E1-E7 in order. Each exercise includes the explanation, code and completion check needed to follow it. Before changing an assumption, write a prediction; afterwards, compare it with the result. The optional reference section at the end provides more detail.

Logging, plots and file handling are supplied throughout; the lecture demonstration is ready to run.
Coding centres on E1/E3 and short policy functions in E4/E7; the other exercises focus on choices and interpretation.

| Exercise | Minutes | Teams implement or choose | Supplied | Data |
| --- | --- | --- | --- | --- |
| E1 | 25 | Track simulation, filter and pull analysis; AI assistance welcome. | Equations and a complete reference alternative. | Simulation with known truth. |
| E2 | 45 | Assess bias/noise, choose a correction and interpret normalization. | Capture, calibration calculations and held-out comparison. | Live stationary IMU preferred; checked or simulated pair available. |
| E3 | 50 | Four prediction/gain/update expressions; choose Q/V. | Model, residuals, covariance update, checks and comparisons. | Live/saved acceleration and ToF; simulation fallback. |
| E4 | 25 | Chi-square threshold function and rejection probability. | Fault injection and gated/ungated runs using your E3 filter. | E3 data replay with an injected range outlier. |
| E5 | 30 | Gap/rest intervals and uncertainty; derive the velocity constraint. | Your E3 functions plus supplied velocity-constraint update. | E3 replay for range loss; simulation for valid/false rest constraints. |
| E6 | 15 | Select independent input; predict and assess the frozen filter. | Capture/replay, frozen calibration/settings and diagnostics. | Fresh live/saved data preferred; independent simulation available. |
| E7 | 25 | Clearance-to-speed rule and margins. | Covariance projection, gate geometry and baseline decisions. | E3/E5 covariance; illustrative geometry and assumed horizontal uncertainty. |

Live acquisition is motor-free. E6's simulation checks implementation consistency; fresh measured data test transfer to the actual drone.

In Jupyter, select **Kernel > Change Kernel > Crazyflie lectures (.venv)**, run setup once, then use Shift+Enter for subsequent cells. The default path uses simulation. Live capture is an explicit choice, with handling and firmware-stop checks beside each recording cell.

E2 checks stationary IMU calibration. E3-E5 reuse one timestamped accelerometer/ToF recording; E6 checks frozen choices on fresh data. If acquisition stalls, a checked sensor recording or simulation lets you continue; spend at most five minutes troubleshooting.

Choose a fresh `run_id` for a new session. Repeated comparisons within a session get numbered output folders. Save the edited notebook and one useful comparison in your team's existing notes.


### Team radio

Set `uri` once in setup, through `CF_URI` or by entering the assigned URI. Match it to the label on your team's drone. Both recording cells reuse this value; close cfclient before connecting from Python.


In [ ]:
from pathlib import Path
import json
import hashlib
import os
import sys
from dataclasses import replace

# Work from the project root, code/, or this notebook's own folder.
root = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "code/day2/experiments.py").is_file())
sys.path.insert(0, str(root / "code"))

import numpy as np
from IPython.display import Image, display
from kalman.filter import FilterConfig, PositionVelocityKalmanFilter
from race_common.dynamics import position_velocity_model
from day2 import vertical
from day2.notebook_tools import (
    capture, capture_vertical, completed_capture, read_imu, plot_imu_calibration, simulated_imu_pair,
    load_capture_overview, plot_capture_overview, display_table,
    next_attempt,
)

run_id = "trial-wh"  # Change before repeating the notebook.
output = root / "day2/notebook" / run_id
output.mkdir(parents=True, exist_ok=True)
seed = 0
uri = os.environ.get("CF_URI", "")  # Or enter your team\'s assigned radio URI here.
print("Python:", sys.executable)
print("Results:", output)


### Position measurements reveal velocity

Start with state $(x,v)$ and position observations only.
The velocity correction is $\Delta v=C_{vx}^{pred}r/(C_{xx}^{pred}+V)$.
Inspect the update; replace time/velocity with z/slope for E1.


In [ ]:
# Exact constant-speed motion: uncertainty comes from initial state and position noise.
demo_time = np.arange(21) * 0.2  # seconds
demo_true_velocity = 0.15  # m/s
demo_true_position = 0.2 + demo_true_velocity * demo_time
demo_position_std = 0.03  # metres
demo_V = demo_position_std**2
demo_rng = np.random.default_rng(7)
demo_measurements = demo_true_position + demo_rng.normal(0, demo_position_std, len(demo_time))

def position_velocity_update(predicted, predicted_C, measurement, V):
    H = np.array([1.0, 0.0])  # The sensor observes position.
    residual = measurement - H @ predicted
    R = H @ predicted_C @ H + V
    K = predicted_C @ H / R
    corrected = predicted + K * residual
    # Joseph form keeps the covariance symmetric and positive under roundoff.
    A = np.eye(2) - np.outer(K, H)
    corrected_C = A @ predicted_C @ A.T + np.outer(K, K) * V
    return corrected, corrected_C, K, residual

demo_state = np.zeros(2)  # Initially, position and velocity are unknown.
demo_C = np.diag([1.0, 1.0])  # Variances: m^2 and (m/s)^2.
demo_states, demo_covariances, demo_updates = [], [], []
for index, measurement in enumerate(demo_measurements):
    dt = 0.0 if index == 0 else demo_time[index] - demo_time[index - 1]
    F = np.array([[1.0, dt], [0.0, 1.0]])
    # Q=0 for this exact constant-velocity example.
    predicted = F @ demo_state
    predicted_C = F @ demo_C @ F.T
    # F creates C_vx: a velocity error also becomes a position error over dt.
    demo_state, demo_C, K, residual = position_velocity_update(
        predicted, predicted_C, measurement, demo_V)
    demo_updates.append(dict(predicted=predicted, C=predicted_C, K=K, residual=residual))
    demo_states.append(demo_state.copy())
    demo_covariances.append(demo_C.copy())
demo_states = np.asarray(demo_states)
demo_covariances = np.asarray(demo_covariances)

demo_update = demo_updates[1]  # First update after a nonzero time step.
demo_delta_v = demo_update["K"][1] * demo_update["residual"]
demo_without_cross_covariance = demo_update["C"].copy()
demo_without_cross_covariance[0, 1] = demo_without_cross_covariance[1, 0] = 0.0
demo_uncoupled_state, _, _, _ = position_velocity_update(
    demo_update["predicted"], demo_without_cross_covariance,
    demo_measurements[1], demo_V)
print(f"First observation (t=0): velocity gain = {demo_updates[0]['K'][1]:.3f} s^-1")
print(f"Next prediction: C_vx = {demo_update['C'][1, 0]:.4f} m^2/s")
print(f"Position residual = {demo_update['residual']:.4f} m; K_v = {demo_update['K'][1]:.4f} s^-1")
print(f"Velocity correction = K_v * residual = {demo_delta_v:.4f} m/s")
print(f"Same update with C_vx=0: velocity correction = "
      f"{demo_uncoupled_state[1] - demo_update['predicted'][1]:.4f} m/s")


In [ ]:
import matplotlib.pyplot as plt

demo_figure, demo_axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
demo_axes[0].plot(demo_time, demo_true_position, "k--", label="True position")
demo_axes[0].plot(demo_time, demo_measurements, ".", color="C1", label="Position measurements")
demo_axes[0].set_ylabel("Position [m]")
demo_axes[1].axhline(demo_true_velocity, color="black", linestyle="--", label="True velocity")
demo_axes[1].set_ylabel("Velocity [m/s]")
for component, axis in enumerate(demo_axes):
    sigma = np.sqrt(demo_covariances[:, component, component])
    axis.plot(demo_time, demo_states[:, component], color="C0", label="Filtered estimate")
    axis.fill_between(demo_time, demo_states[:, component] - 1.96*sigma,
                      demo_states[:, component] + 1.96*sigma,
                      color="C0", alpha=0.2, label="95% model band")
    axis.legend(fontsize="small")
    axis.grid(alpha=0.25)
demo_axes[1].set_xlabel("Time [s]")
demo_figure.tight_layout()
plt.show()


## E1: follow a particle track (25 minutes)

Ten parallel detector planes lie at $z=1,\ldots,10$ m, each 1 m tall in x.
Gaussian position resolution: $\sigma=0.01$ m. Each plane causes an independent
Gaussian slope kick of standard deviation 0.02: local Q adds $0.02^2$ to slope
variance, leaving position unchanged.

1. **Simulate tracks.** Choose starting x and slope at z = 0; for example,
   independent zero-mean Gaussians with standard deviations 0.1 m and 0.1.
   Propagate with scattering and generate noisy measurements.
2. **Write a Kalman filter**, state (x, slope), with an initial state
   and large covariance. Predict, then update at each plane. Measure before
   scattering; estimate the incoming state and covariance C at the last plane.
3. **Run 1,000 tracks.** Histogram both pulls,
   $(\hat x_i-x_i^{true})/\sqrt{C_{ii}}$. Fit or report means and standard
   deviations; are they consistent with $N(0,1)$?

Use Python, NumPy and Matplotlib; notation: state x, covariance C, transport F,
process noise Q, projection H, measurement variance V, gain K.
[Slides](https://www.nikhef.nl/~wouterh/teaching/TrackingAndVertexing/index.html).

We suggest feeding this prompt into your favourite AI.
The most beautiful illustration before noon wins a **pakje stroopwafels**.

**Completion:** explain one predict/update and keep code, seed, figure and pull statistics.

Reference: optional cell below; includes virtual hits outside the active height.


In [ ]:
# Try your implementation first, or enable the supplied reference for comparison.
run_tracking_reference = False
if run_tracking_reference:
    from day2 import tracking
    detector = tracking.Detector(scattering_slope_std=0.02)
    tracking_result = tracking.ensemble(tracks=1000, seed=7, detector=detector)
    tracking_output = next_attempt(output / "tracking")
    tracking.save(tracking_result, tracking_output)
    print(json.dumps(tracking_result["summary"], indent=2))
    display(Image(filename=str(tracking_output / "track.png")))
    display(Image(filename=str(tracking_output / "pulls.png")))


## E2: check calibration before tuning — 45 minutes

Record the accelerometer and gyroscope channels of the **inertial measurement unit (IMU)** while the drone rests on a stable surface. Use two separate 60-second files: one to estimate gyro offsets and variability, and one to check that correction. Accelerometer readings are body-frame specific force in g; gyro readings are degrees/s and may already include onboard compensation.

Choose **one** recording route: cfclient (E2A) or the notebook (E2B). One laptop connects to the drone; share the saved files within the team. Allow about 5 minutes to identify channels, 10 to record, 15 to calculate, 10 for the held-out check and 5 to save the result.

### E2A: acquire the IMU files with cfclient

1. Place the drone on a stable, undisturbed surface. Connect to the assigned URI. In **Parameters > stabilizer > stop**, enter `1`, press Enter and check the displayed value. Leave the stop engaged and keep clear of propellers.
2. In **Settings > Logging configuration > Default > Create config**, create an IMU block with six float channels: `acc.x`, `acc.y`, `acc.z`, `gyro.x`, `gyro.y`, `gyro.z`. Set **Logging period** to **20 ms** and save; check names/types in **Log TOC**. Create a Motors block with `motor.m1`, `motor.m2`, `motor.m3`, `motor.m4` at 50 ms if needed.
3. In **Log Blocks**, start Motors and check all four traces are zero in **Plotter**. Enable **Start** and **Write to file** for IMU. Record 60 seconds at rest, stop writing, then make a **separate 60-second validation recording** in the same pose.
4. Use **Settings > Open config folder** to find `logdata/` and its timestamped recordings. Retain both CSVs and note firmware, decks and conditions. Disconnect cfclient before any Python capture.

### E2B: record the IMU files with Python

Place the drone on a stable surface, keep clear of propellers and close other radio clients. Set `ready_for_motor_stop = True` and `run_imu_capture = True` below. The helper sets `stabilizer.stop=1` and verifies the firmware lock and zero motor outputs before recording. Leave the drone at rest.

Run once with `capture_filename = "calibration.csv"`, then again with `"validation.csv"`. Each file records 60 seconds after the first IMU packet, at a requested 20 ms period, in `day2/notebook/<run_id>/E2B/`. Existing files are preserved; choose a new name or run ID to repeat. Partial captures remain available for diagnosis but are excluded from automatic selection.

**After either route:** the stop remains engaged until power-cycle. Place the drone down, power-cycle it and repeat preflight checks before flight. If acquisition blocks progress, follow the five-minute fallback rule above; mark calibration unfinished when using the supplied filter assumptions.


In [ ]:
run_imu_capture = False  # Opt in for a stationary recording.
ready_for_motor_stop = False  # Drone on surface; propellers clear; other radio clients closed.
capture_filename = "calibration.csv"  # Then validation.csv for a separate recording.
channels = (
    "acc.x", "acc.y", "acc.z",
    "gyro.x", "gyro.y", "gyro.z",
)

if run_imu_capture:
    if not ready_for_motor_stop:
        raise ValueError("Confirm the drone is on a stable surface and other radio clients are closed.")
    saved_imu, sample_count = capture(
        uri, output / "E2B" / capture_filename,
        channels=channels, duration_s=60, period_ms=20)
    print(f"Saved {sample_count} IMU samples to {saved_imu}")
else:
    print("IMU capture skipped. Select checked stationary files below, or continue offline.")


### E2C: calculate and validate

Select `calibration_csv` and a separate `validation_csv`; completed, unchanged E2B files are selected automatically. Enter checked cfclient paths explicitly. Set `stationary_checks_passed = True` after checking the firmware stop, zero motor outputs and undisturbed rest.

If acquisition stalls, `use_simulated_imu = True` below lets you continue; spend at most five minutes troubleshooting. This creates `E2-simulated/calibration.csv` and `validation.csv` inside this run's output folder, with independent noise and labelled simulation receipts. Complete the same bias, gravity and held-out checks; record calibration of your actual drone as outstanding.

At rest, the expected gyro reading is approximately zero, so its mean estimates an offset. Hold this estimate fixed when checking the validation file.
Before plotting, predict the mean and RMS of $(y-\bar y_{cal})/s_{cal}$, with $s_{cal}^2=N^{-1}\sum(y-\bar y_{cal})^2$. For calibration samples they are **zero and one by construction**, even for a non-Gaussian sensor. The histogram still reveals shape and tails. The held-out histogram uses the **same frozen calibration mean and RMS**: its mean and width test whether the correction and noise scale transfer. For acceleration, this comparison also assumes the same stationary orientation. Shared systematic errors can remain hidden.

Within the existing held-out comparison, explain why only one histogram is forced to have zero mean and unit RMS, then interpret one channel. The dashed N(0,1) curve is a reference shape, not a fitted validation result.

The accelerometer measures specific force: at rest, its components reflect gravity projected onto the body axes. Tilt can therefore produce X/Y readings. Check that the magnitude is about **1 g**; separating accelerometer bias from tilt requires additional known orientations.

**Completion:** explain the gyro correction or retained compensation using the held-out comparison, and show the gravity check. Longer warm-up, multi-pose accelerometer, flow, range and Lighthouse studies are optional background.


In [ ]:
calibration_csv = None  # Or select a checked stationary IMU CSV.
validation_csv = None   # A separate stationary recording.
stationary_checks_passed = False  # After the motor-stop and undisturbed-rest checks.
use_simulated_imu = False  # Explicit E2 fallback; creates a labelled pair below.

# Invalidate earlier results before trying another pair of files.
channel_values = validation_values = None
gyro_bias = gravity_norm_g = corrected = calibration_summary = None
standardized_calibration = standardized_validation = None
calibration_source = "selected recordings"
if use_simulated_imu:
    calibration_csv, validation_csv = simulated_imu_pair(output / "E2-simulated")
    calibration_source = "simulation; calibration of the actual drone remains outstanding"
print("Calibration source:", calibration_source)
if calibration_csv is None:
    calibration_csv = completed_capture(output / "E2B" / "calibration.csv")
if validation_csv is None:
    validation_csv = completed_capture(output / "E2B" / "validation.csv")

if (stationary_checks_passed or use_simulated_imu) and calibration_csv is not None:
    if validation_csv is not None and Path(calibration_csv).samefile(validation_csv):
        raise ValueError("Calibration and held-out validation need two separate recording files.")
    training_values = read_imu(calibration_csv)
    if validation_csv is not None:
        validation_values = read_imu(validation_csv)
    channel_values = training_values  # Publish the selection only after both files load.
else:
    print("Calibration skipped. Select a checked IMU file and confirm the stationary checks.")


In [ ]:
gyro_bias = gravity_norm_g = None
standardized_calibration = standardized_validation = None
if channel_values is not None:
    channel_names = tuple(f"{sensor}.{axis}" for sensor in ("acc", "gyro") for axis in "xyz")
    acceleration = channel_values[:, :3]  # Body specific force [g].
    training = np.deg2rad(channel_values[:, 3:])
    gyro_bias = training.mean(axis=0)
    print("Calibration mean [rad/s]:", gyro_bias)
    print("Calibration covariance [(rad/s)^2]:", np.cov(training, rowvar=False, ddof=1))
    print("Calibration sample std [rad/s]:", training.std(axis=0, ddof=1))
    midpoint = len(training) // 2
    print("First/second-half means [rad/s]:",
          training[:midpoint].mean(axis=0), training[midpoint:].mean(axis=0))

    # Compare samples directly: mean-rounding can give constant decimals a nonzero RMS.
    varying = np.any(channel_values != channel_values[0], axis=0)
    channel_means = channel_values.mean(axis=0)
    channel_means[~varying] = channel_values[0, ~varying]
    centered = channel_values - channel_means
    sigma = np.sqrt(np.mean(centered**2, axis=0))
    standardized_calibration = np.full_like(centered, np.nan)
    standardized_calibration[:, varying] = centered[:, varying] / sigma[varying]
    if validation_values is not None:
        # Freeze both quantities from calibration; never recenter or rescale held-out data.
        standardized_validation = np.full_like(validation_values, np.nan)
        standardized_validation[:, varying] = (validation_values[:, varying] - channel_means[varying]) / sigma[varying]
        print("Held-out standardized mean/std (varying channels):",
              standardized_validation[:, varying].mean(axis=0),
              standardized_validation[:, varying].std(axis=0))
    if not np.all(varying):
        print("Channels with zero measured variation:",
              [name for name, valid in zip(channel_names, varying) if not valid])
    gravity_norm_g = np.linalg.norm(acceleration, axis=1)
    print("Stationary specific-force magnitude: mean/std [g]:",
          gravity_norm_g.mean(), gravity_norm_g.std(ddof=1))


In [ ]:
if gyro_bias is not None:
    display_table([
        {"channel": name, "mean": float(mean), "RMS sigma": float(spread),
         "unit": "g" if name.startswith("acc.") else "deg/s"}
        for name, mean, spread in zip(channel_names, channel_means, sigma)
    ])
    plot_imu_calibration(channel_values, channel_names, standardized_calibration, standardized_validation)


In [ ]:
corrected = None
if gyro_bias is not None and validation_values is not None:
    import matplotlib.pyplot as plt

    validation = np.deg2rad(validation_values[:, 3:])
    corrected = validation - gyro_bias  # Keep the training bias fixed.
    print("Held-out mean before correction [rad/s]:", validation.mean(axis=0))
    print("Held-out mean after correction [rad/s]:", corrected.mean(axis=0))
    print("Held-out std before/after [rad/s]:",
          validation.std(axis=0, ddof=1), corrected.std(axis=0, ddof=1))
    figure, axes = plt.subplots(3, 1, figsize=(9, 6), sharex=True)
    for index, axis in enumerate(axes):
        axis.plot(validation[:, index], label="held-out")
        axis.plot(corrected[:, index], label="training bias subtracted")
        axis.set_ylabel(f"gyro {'xyz'[index]} [rad/s]")
    axes[0].legend()
    axes[-1].set_xlabel("sample index in validation recording")
    figure.tight_layout()
    plt.show()
elif gyro_bias is not None:
    print("Select a separate validation recording to complete the held-out check.")


### Save the calibration decision

Enter your interpretation below; the summary saves the results for later discussion. It records an analysis correction, while the drone retains its existing calibration. E3 estimates range variability during its own stationary check; the gyro result is kept separate.


In [ ]:
calibration_decision = "unfinished"  # Replace with your held-out comparison and decision.
calibration_summary = None
if gyro_bias is not None:
    calibration_summary = {
        "decision": calibration_decision,
        "source": calibration_source,
        "gyro_bias_rad_s": gyro_bias.tolist(),
        "gyro_covariance_rad2_s2": np.cov(training, rowvar=False, ddof=1).tolist(),
        "gravity_mean_g": float(gravity_norm_g.mean()),
        "validation_completed": corrected is not None,
        "held_out_corrected_mean_rad_s": None if corrected is None else corrected.mean(axis=0).tolist(),
        "source_sha256": {
            str(Path(path).resolve()): hashlib.sha256(Path(path).read_bytes()).hexdigest()
            for path in (calibration_csv, validation_csv) if path is not None
        },
    }
    calibration_output = next_attempt(output / "E2")
    calibration_output.mkdir()
    (calibration_output / "calibration.json").write_text(
        json.dumps(calibration_summary, indent=2) + "\n", encoding="utf-8")
    print("Saved calibration decision:", calibration_output / "calibration.json")


## From the opening demonstration to sensor fusion

The state is now height and vertical velocity. The accelerometer supplies the process input; the downward ToF sensor measures distance to a fixed floor. A level drone moving gently up and down makes range approximately height. Subtract the initial stationary `acc.z` mean, including gravity, and multiply by 9.80665 to obtain approximate vertical acceleration. This retains existing onboard sensor compensation and uses no onboard position, velocity or attitude estimate.

Both channels carry firmware log timestamps in the same packet. The helper unwraps milliseconds to seconds, uses actual dt, and rejects duplicate/reversed times or gaps above 0.30 s. Log timestamps approximate acquisition time; channel delays and internal filtering remain model limitations. ToF range is floor-relative, rather than Lighthouse/world height.


## Capture one sensor recording, or use simulation

A Flow deck with a working downward ToF sensor is needed for live data; Lighthouse positioning is not used. Close cfclient and other radio clients. Place the drone on a stable, level surface with a clear view of a fixed flat floor and keep hands clear of the propellers. Set the assigned URI and opt in below.

Use a stable raised support with an opening beneath the sensor, 5-20 cm above the floor. The ToF beam must see the same fixed floor during the initial rest and subsequent motion.

The helper sets `stabilizer.stop=1`, verifies the firmware lock and zero motor outputs, and keeps monitoring them. **Leave the drone on the surface until STOP VERIFIED.** Then keep it stationary and level for five seconds. After **Calibration check passed**, move it gently up and down while keeping the beam on the same floor. Hold still again by the 15-second prompt. Capture ends at 20 seconds; note which rest intervals actually occurred.

The stop stays engaged after success, interruption or failure. Place the drone down and power-cycle before normal flight/preflight checks. Connection plus capture is bounded to 50 seconds. If acquisition or model checks stall, a checked sensor recording or simulation lets you continue; spend at most five minutes troubleshooting and retain unexpected data for later investigation.


In [ ]:
run_live_capture = False
ready_for_motor_stop = False  # True after placing the drone down and closing radio clients.
capture_name = "capture-01"  # Choose a new name for another recording.
captured_directory = None  # Invalidate any previous capture before a new attempt.
if run_live_capture:
    data = settings = training_calibration = None
    baseline_result = dropout = validation_data = validation_result = None
    captured_directory = capture_vertical(
        output / capture_name, uri, ready_for_motor_stop=ready_for_motor_stop, duration_s=20)
else:
    print("Capture skipped. Explicitly choose saved sensor data or simulation below.")


## Choose data once

Set `recording_directory` to a complete sensor recording containing `raw.csv` and `recording.json`, or explicitly leave it `None` for the simulated rest/up/down/rest experiment. This format has timestamped acceleration and range, not onboard position estimates. Files are hash-checked; invalid selections are errors, not silent fallback.

Selection clears the previous dataset before loading. The five-second rest calibration precedes filtering, so later motion never contributes to its fitted offset. All E3/E4/dropout comparisons reuse the same input and calibration. Simulation supplies truth for RMSE/coverage; a real recording has no independent truth.


In [ ]:
data = None
baseline_result = None
dropout = None
validation_result = None
settings = None
training_calibration = None
recording_directory = captured_directory  # Or: root / "data/vertical-training"; None selects simulation.
data = vertical.select(recording_directory, seed=seed)
training_calibration = data.calibration
settings = vertical.Settings(range_std_m=training_calibration.range_std_m)
print("Source:", data.metadata["source"], "input hash:", data.metadata["input_sha256"])
print("Calibration:", training_calibration)
print("Filtering timestamps:", data.time_s[0], "to", data.time_s[-1], "seconds")


## E3: construct the filter, then change its assumptions — 50 minutes

Allow 25 minutes for data selection/capture and the four expressions below (at most five minutes on acquisition), 20 for matched comparisons and 5 for discussion. Logging and plotting are supplied.

For $x=(z,v_z)^T$, $H=(1,0)$ and each measured time interval:
$$F=\begin{pmatrix}1&\Delta t\\0&1\end{pmatrix},\quad B=\begin{pmatrix}\tfrac12\Delta t^2\\\Delta t\end{pmatrix},\quad Q=BB^T\sigma_a^2.$$
$$x^-=Fx+Ba,\quad C^-=FCF^T+Q,\quad r=m-Hx^-,\quad R=HC^-H^T+V,$$
$$K=C^-H^T/R,\quad x^+=x^-+Kr.$$
The Joseph covariance update is provided. Compare with `vertical.predict` and `vertical.update` if you need a worked reference. This uses an independent unknown acceleration perturbation held over each interval; its Q has a different sample-time dependence from continuous white-acceleration noise.

**Before running:** predict the signs of both corrections for a positive height residual. Explain why a position observation changes velocity: the velocity component of K is $C^-_{vz}/R$. Its units are 1/s. Set that cross-covariance to zero for one update and predict what disappears.

Replace the four `None` expressions. Unfinished expressions stop with a clear message; the checks must pass before the recording is processed. Trace one timestamped prediction/update, printing K, the correction and covariance. Try a different dt and V.

Then compare integration of acceleration, differences of range and the KF. Change V by factors 0.1 and 10 while keeping measurements fixed. Choose one acceleration standard deviation for Q. Inspect height, velocity, ±1.96-sigma bands and **pre-gate** pull widths. Under an unbiased consistent Gaussian model, scalar pulls have width near one; simulation truth supports **95%** marginal coverage checks. Smoothness alone does not establish accuracy.

**Completion:** calculate one update, explain velocity correction through cross-covariance, and retain one V comparison plus a justified Q choice. No independent accuracy claim follows from real data without a separate reference.


In [ ]:
def our_predict(x, C, acceleration, dt, acceleration_std):
    F, B = vertical.model(dt)
    Q = np.outer(B, B) * acceleration_std**2
    x_predicted = None  # TODO E3: propagate the state.
    C_predicted = None  # TODO E3: propagate covariance and add Q.
    if x_predicted is None or C_predicted is None:
        raise NotImplementedError("Complete the two prediction expressions.")
    return x_predicted, C_predicted

def our_update(x_predicted, C_predicted, measurement, V):
    H = np.array([1., 0.])
    residual = measurement - H @ x_predicted
    R = H @ C_predicted @ H + V
    K = None  # TODO E3: gain from covariance and residual variance.
    x_filtered = None  # TODO E3: correct the predicted state.
    if K is None or x_filtered is None:
        raise NotImplementedError("Complete the gain and state-correction expressions.")
    A = np.eye(2) - np.outer(K, H)
    C_filtered = A @ C_predicted @ A.T + np.outer(K, K) * V
    return x_filtered, C_filtered

vertical.check_steps(our_predict, our_update)


In [ ]:
x = np.array([0.3, 0.0])
C = np.diag([0.02**2, 0.10**2])
dt = data.time_s[1] - data.time_s[0]
x_predicted, C_predicted = our_predict(x, C, data.acceleration_m_s2[1], dt,
                                      settings.acceleration_std_m_s2)
measurement = x_predicted[0] + 0.01  # A specified +1 cm predicted residual.
K = C_predicted[:, 0] / (C_predicted[0, 0] + settings.range_std_m**2)
x_filtered, C_filtered = our_update(x_predicted, C_predicted, measurement, settings.range_std_m**2)
print("dt [s], K [dimensionless, 1/s]:", dt, K)
print("Correction [m, m/s]:", x_filtered - x_predicted)
print("C before/after:\n", C_predicted, "\n", C_filtered)


In [ ]:
baseline_settings = vertical.Settings(range_std_m=training_calibration.range_std_m)
candidate_acceleration_std = 0.08  # Choose and justify; observations stay fixed.
chosen_name = "Q-candidate"  # Select one matched case after inspecting the comparison.


In [ ]:
if data is None:
    raise ValueError("Rerun the data-selection cell successfully before continuing.")
baseline_result = vertical.run(data, baseline_settings, predict_step=our_predict, update_step=our_update)
cases = {
    "baseline": baseline_result,
    "V-small": vertical.run(data, replace(baseline_settings, range_std_m=baseline_settings.range_std_m * np.sqrt(0.1)),
                            predict_step=our_predict, update_step=our_update),
    "V-large": vertical.run(data, replace(baseline_settings, range_std_m=baseline_settings.range_std_m * np.sqrt(10)), predict_step=our_predict, update_step=our_update),
    "Q-candidate": vertical.run(data, replace(baseline_settings, acceleration_std_m_s2=candidate_acceleration_std), predict_step=our_predict, update_step=our_update),
}
e3_output = next_attempt(output / "E3")
display(Image(filename=str(vertical.save_comparison(data, cases, e3_output))))
display_table([dict(case=name, **vertical.summary(data, result)) for name, result in cases.items()])
settings = vertical.Settings(**cases[chosen_name]["settings"])
baseline_result = cases[chosen_name]
print("Frozen training choice:", chosen_name, settings)


## E4: one anomalous measurement — 25 minutes

A scalar height predicted residual has one degree of freedom. Replace `None` in `our_residual_threshold` using `chi2.ppf(1-alpha, df=dimension)`, and explain the chosen false-rejection probability. The default alpha is 0.01.

Predict the state correction from a **0.15 m** added range observation at/after **12 s**, then compare ungated and gated runs on that identical copy. Inspect the saved residual, its variance, chi-square, prospective gain, acceptance and recovery. The original recording is unchanged.

A large chi-square can also indicate a poor propagation model, timing error or changed floor geometry. Rejection is not a substitute for understanding those cases. Count rejections on clean data as well; retain rejected residuals in the diagnostic distribution.

**Completion:** one explained rejection and one reason a valid measurement might be rejected. More fault scenarios are optional.


In [ ]:
from scipy.stats import chi2

false_rejection_probability = 0.01  # Choose and justify.

def our_residual_threshold(dimension, alpha):
    # TODO E4: replace None with the dimension-aware threshold.
    return None



In [ ]:
if data is None:
    raise ValueError("Rerun the data-selection cell successfully before continuing.")
fault_data = vertical.with_outlier(data, time_s=12, offset_m=0.15)
ungated = vertical.run(fault_data, settings, predict_step=our_predict, update_step=our_update)
gated = vertical.run(fault_data, settings, predict_step=our_predict, update_step=our_update,
                     residual_gate=our_residual_threshold, alpha=false_rejection_probability)
clean_gated = vertical.run(data, settings, predict_step=our_predict, update_step=our_update,
                           residual_gate=our_residual_threshold,
                           alpha=false_rejection_probability)
print("Clean:", vertical.summary(data, clean_gated))
print("Fault, gated:", vertical.summary(fault_data, gated))
e4_clean_output = next_attempt(output / "E4-clean")
display(Image(filename=str(vertical.save_comparison(
    data, {"ungated": baseline_result, "gated": clean_gated}, e4_clean_output))))
e4_output = next_attempt(output / "E4")
display(Image(filename=str(vertical.save_comparison(
    fault_data, {"ungated": ungated, "gated": gated}, e4_output))))


## E5: remove information, then add a justified constraint — 30 minutes

Spend 15 minutes on the missing-range comparison and 15 on a known-rest constraint.

Suppress range updates during **8-11 s**, retaining timestamped acceleration inputs. Predict the height/velocity uncertainty growth and the correction when range returns. The radio and onboard estimator are unchanged. Compare the covariance and gain before, during and after the gap.

Quiet IMU readings also occur during constant-speed translation; rest needs independent evidence.

A known stationary interval gives a velocity observation $m=0$, $H=(0,1)$, with **finite** variance. It is a measurement update, not an instruction to overwrite velocity and set its variance to zero. Derive the gain and expected covariance change before running it.

Use the simulated rest/motion recording for the constraint comparison: rest is specified at 15-20 s. Choose the velocity standard deviation and a subinterval. Then deliberately apply the same constraint during simulated motion and explain the bias. This controlled counterexample needs no second hardware capture.
All three runs reuse your E3 prediction and range-update functions; the helper supplies the velocity-constraint update.

**Completion:** explain uncertainty growth without range and the effect of a valid versus false zero-velocity update. Designing an automatic stationary detector, including `duration_s` and fresh `flow_speed_m_s` evidence, is optional sensor-characterization work.


In [ ]:
if data is None:
    raise ValueError("Rerun the data-selection cell successfully before continuing.")
if baseline_result is None:
    raise ValueError("Rerun E3 on the selected data before continuing.")
range_gap = (8.0, 11.0)
dropout = vertical.run(data, settings, predict_step=our_predict, update_step=our_update, range_gap=range_gap)
e5_dropout_output = next_attempt(output / "E5-dropout")
display(Image(filename=str(vertical.save_comparison(
    data, {"baseline": baseline_result, "range-gap": dropout}, e5_dropout_output))))
gap_index = np.flatnonzero(data.time_s <= range_gap[1])[-1]
print("At gap end, baseline C:\n", baseline_result["covariance"][gap_index])
print("At gap end, dropout C:\n", dropout["covariance"][gap_index])
recovery_index = np.flatnonzero(dropout["accepted"] & (data.time_s > range_gap[1]))[0]
print("First returning range at [s]:", data.time_s[recovery_index])
print("Predicted residual, variance, gain:", dropout["predicted_residual"][recovery_index],
      dropout["predicted_residual_variance"][recovery_index], dropout["gain"][recovery_index])


In [ ]:
rest_settings = replace(settings, stationary_velocity_std_m_s=0.02)  # Choose and justify.
known_rest = [(16.0, 19.0)]
false_rest = [(8.0, 9.0)]  # Deliberate simulation-only counterexample.


In [ ]:
rest_data = vertical.select(seed=seed)  # Known simulated rest; truth never enters the update.
rest_runs = {
    "without-constraint": vertical.run(rest_data, rest_settings, predict_step=our_predict, update_step=our_update),
    "known-rest": vertical.run(rest_data, rest_settings, predict_step=our_predict, update_step=our_update, stationary_intervals=known_rest),
    "false-rest": vertical.run(rest_data, rest_settings, predict_step=our_predict, update_step=our_update, stationary_intervals=false_rest),
}
e5_rest_output = next_attempt(output / "E5-rest")
display(Image(filename=str(vertical.save_comparison(rest_data, rest_runs, e5_rest_output))))
for name, result in rest_runs.items():
    print(name, vertical.summary(rest_data, result))


## E6: frozen settings on fresh data — 15 minutes

Allow five minutes to select a new capture or supplied recording, five to compare and five to discuss. Freeze Q/V and the rest calibration chosen in E3. To capture here, opt in with `run_validation_capture` and confirm the same handling checks with `ready_for_validation_motor_stop`; choose a fresh output name if repeating. The capture checks initial rest, but the filter keeps the training correction. Stop troubleshooting after five minutes. Setting `validation_directory=None` explicitly selects independent simulated noise with seed +20, labelled as simulation.

The validation recording must differ from training: identical input hashes are rejected. The helper retains timestamps and the 0.30 s gap limit, and starts from the first height observation and configured initial velocity uncertainty. It does not refit the stationary bias or V. Explain any startup difference before interpreting later behaviour.
The simulated fallback uses the frozen stationary correction and declared acceleration/range noise settings to generate independent samples. This checks implementation consistency under those assumptions; a fresh real recording tests their transfer to the drone.

**Completion:** one prediction checked on fresh data, one limitation and a decision about whether the frozen assumptions remain credible. Record experimental validation as outstanding when only simulation was used.


In [ ]:
validation_data = None
validation_result = None
if data is None:
    raise ValueError("Rerun the data-selection cell successfully before continuing.")
validation_data = None  # A failed selection must not leave the previous validation input active.
validation_directory = None  # A distinct capture/supplied sensor directory, or explicit simulation.
run_validation_capture = False
ready_for_validation_motor_stop = False  # Confirm the drone is down, hands clear and radio clients closed.
if run_validation_capture:
    validation_directory = capture_vertical(
        output / "validation-capture-01", uri, duration_s=20,
        ready_for_motor_stop=ready_for_validation_motor_stop)
validation_data = vertical.select(validation_directory, seed=seed + 20, calibration=training_calibration, settings=settings)
if validation_data.metadata["input_sha256"] == data.metadata["input_sha256"]:
    validation_data = None
    raise ValueError("Choose a different training/validation recording.")
validation_result = vertical.run(validation_data, settings, predict_step=our_predict, update_step=our_update)
e6_output = next_attempt(output / "E6")
display(Image(filename=str(vertical.save_comparison(
    validation_data, {"frozen-settings": validation_result}, e6_output))))
print("Validation source:", validation_data.metadata["source"])
print(vertical.summary(validation_data, validation_result))


## E7: use the filter's covariance in a race decision — 25 minutes

Take **actual C** at the same timestamp from E3 and the E5 range-gap run. The height variance C[0,0] supplies the vertical entry of the three-dimensional position covariance below. Horizontal uncertainties are explicitly assumed for this 1D experiment; it has not estimated X/Y motion.

Project with $C_{plane}=U^T C_{pp}U$, where U spans the gate plane. Compare normals [1,0,0] (vertical opening) and [0,0,1] (horizontal opening). Which opening is affected by growing height uncertainty? Uncertainty along the normal should not directly reduce in-plane clearance.

Clearance is opening radius minus radial offset, drone radius, tracking margin and k times the worst in-plane sigma. Defaults are illustrative; the sigma allowance is not an exact collision probability. Use the computed baseline/dropout values to justify a candidate speed rule, including **zero forward speed for non-positive clearance**. State the approach distance and braking assumptions before proposing physical use.

**Completion:** retain the two C projections and speed decisions, explaining the chain from missing measurement to uncertainty to action. These are planning decisions; speed recommendations do not change simulated motion or establish a lap-time improvement.

The example transfers uncertainty, rather than a floor-relative height value, into an illustrative gate geometry. Real clearance also needs floor/world reference and survey errors.


In [ ]:
if data is None:
    raise ValueError("Rerun the data-selection cell successfully before continuing.")
if dropout is None:
    raise ValueError("Rerun E5's range-gap comparison before continuing.")
if baseline_result is None:
    raise ValueError("Rerun E3 on the selected data before continuing.")
from race_common.hoop import Hoop
from race_common.strategy import RaceLimits, recommended_speed

limits = RaceLimits()
estimated_position = np.array([1.0, 0.03, 0.7])
horizontal_std_m = 0.04  # Assumed here, not inferred by the vertical filter.

def our_crossing_speed(clearance, limits):
    if clearance <= 0:
        return 0.0
    # TODO E7: choose and justify a slower positive-clearance regime.
    return limits.nominal_speed_m_s

for label, result in (("range available", baseline_result), ("range gap", dropout)):
    C_position = np.diag([horizontal_std_m**2, horizontal_std_m**2,
                          result["covariance"][gap_index, 0, 0]])
    for name, normal in (("vertical", [1, 0, 0]), ("horizontal", [0, 0, 1])):
        gate = Hoop(np.array([1.0, 0.0, 0.7]), np.array(normal), radius_m=0.5)
        speed, clearance = recommended_speed(gate, estimated_position, C_position, limits)
        print(label, name, "projected covariance [m^2]:", gate.plane_covariance(C_position))
        print("Clearance [m]:", clearance, "baseline/candidate [m/s]:",
              speed, our_crossing_speed(clearance, limits))


## Discuss and keep

Save this notebook, the E2 calibration values/decision and one useful comparison. In the team's existing notes, explain one failed prediction, its likely cause and one race-day implication. Label results as live, supplied or simulated. Figures, CSVs and settings carry the detail; no separate report is needed.

The completion checks above define the core. Extra seeds, longer studies, additional calibration and new state models are extensions for spare time. The reference sections below can be consulted as needed.

### Optional background

- [Kalman-filter practical](../../docs/kalman_filter_practical.md): command-line routes and extensions.
- [Sensor recording details](../../docs/sensor_recording_contract.md): formats, timing and firmware references.
- [Calibration background](../../docs/sensor_bias_calibration_exercise.md): broader sensor characterization.
- [Glossary](../../docs/glossary.md): additional terminology.


## Optional: tracking conventions and illustration

**Conventions to agree.** The reference measures first and scatters afterwards.
Its filtered state at the last plane is therefore the incoming state, before
that plane's kick. The kick covariance at a plane is
$Q_{\mathrm{plane}}=\mathrm{diag}(0,0.02^2)$. Over the next gap d,

$$F=\begin{pmatrix}1&d\\0&1\end{pmatrix},\qquad
Q_{\mathrm{gap}}=FQ_{\mathrm{plane}}F^T
=0.02^2\begin{pmatrix}d^2&d\\d&1\end{pmatrix},\qquad
H=(1,0),\qquad V=0.01^2\,\mathrm{m}^2.$$

A kick changes position uncertainty only after propagation. The first gap from
z = 0 to z = 1 contains no scattering plane. The reference starts truth with the
example 0.1 m / 0.1 spreads, and uses a zero-mean fit with initial standard
deviations 1 m / 1. A broad fit prior can affect pulls; check its influence
rather than imposing unit width by rescaling the errors.

**Optional plot challenge:** reproduce the green widening prediction bands, measured hits and dotted initial direction.

The reference below supplies a comparison or fallback. Its all-plane control includes virtual hits outside the 1 m active height and reports the number of fully contained tracks. Finite-aperture mode skips outside hits and retains all attempted tracks; acceptance and missing-hit information affect the pull interpretation.


## Optional: a six-state drone model

We follow R. Frühwirth, [Application of Kalman filtering to track and vertex fitting (1987)](https://doi.org/10.1016/0168-9002(87)90887-4). The state is $x_k=(\mathbf{p}_k^T,\mathbf{v}_k^T)^T$: three positions and three velocities. With acceleration held constant over $\Delta t$,
$$
\mathbf{v}_{k+1}=\mathbf{v}_k+\Delta t\,\mathbf{a}_k,\qquad
\mathbf{p}_{k+1}=\mathbf{p}_k+\Delta t\,\mathbf{v}_k+\tfrac12\Delta t^2\,\mathbf{a}_k.
$$
The example initializes a state and uncertainty, predicts using this model, then updates with a position measurement. Process input is the acceleration; the position observation supplies the measurement update.

State covariance is $C$, measurement covariance is $V$, and residual covariance is $R$. Diagonal entries of C are state-estimation error variances: m² for position and (m/s)² for velocity. Superscript $k-1$ denotes a prediction using measurements through the preceding step:
$$
x_k^{k-1}=F_{k-1}x_{k-1}+B_{k-1}\mathbf{a}_{k-1},\qquad
C_k^{k-1}=F_{k-1}C_{k-1}F_{k-1}^T+Q_{k-1}.
$$
For position, $H=[I,0]$. The predicted residual, its covariance and the gain are
$$
r_k^{k-1}=m_k-H_kx_k^{k-1},\qquad
R_k^{k-1}=V_k+H_kC_k^{k-1}H_k^T,\qquad
K_k=C_k^{k-1}H_k^T(R_k^{k-1})^{-1}.
$$
The update is $x_k=x_k^{k-1}+K_kr_k^{k-1}$. The code uses the Joseph update for C, described in the optional numerical reference below. Its diagnostics expose the predicted residual as `last_predicted_residual`, its covariance as `last_predicted_residual_covariance`, and V as `last_V`. The chi-square increment is $(r_k^{k-1})^T(R_k^{k-1})^{-1}r_k^{k-1}$. The filtered residual $r_k=m_k-H_kx_k$ is evaluated after the update.

**Optional exploration:** reproduce one prediction/update, identify the six states and distinguish input from measurement. Change either `dt`, initial uncertainty or measurement uncertainty, and explain the effect on the estimate and gain. Keep your prediction and result in the notebook.


In [ ]:
dt = 0.02  # seconds
F, B = position_velocity_model(dt)
display(F, B)
np.set_printoptions(precision=3, suppress=True)

# Standard deviations are the square roots of the covariance diagonal.
def to_errors(C):
    return np.sqrt(np.diag(C))

# filter config holds some constants for the kalman filter, in particular the values
# used to fill the covariance matrices
config = FilterConfig(initial_position_std=0.25, position_noise_std=0.04)
# this objects holds the state-vector and the math to perform the filter
estimator = PositionVelocityKalmanFilter(config)
# Initialize position (0, 0, 1) and velocity (0, 0, 0).
# Initial covariance C_0 is set by config above.
estimator.x[2] = 1.0
print(f"Initial state: {estimator.x} +/- {to_errors(estimator.C)}")
# assume an external acceleration
acceleration = [0.2, 0.0, 0.0]
# predict the next state, 
estimator.predict(dt, acceleration)
print(f"Predicted state: {estimator.x} +/- {to_errors(estimator.C)}")
# update the state with an external position measurement.
# Measurement covariance V is set by position_noise_std above.
# Edit position_noise_std in config above to compare measurement uncertainties.
measurement= np.array([0.02, -0.01, 1.0])
estimator.update_position(measurement)
print(f"Measurement: {measurement} +/- {to_errors(estimator.last_V)}")
print(f"Predicted residual: {estimator.last_predicted_residual} +/- {to_errors(estimator.last_predicted_residual_covariance)}")
print(f"Chi-square increment: {estimator.last_chi2:.3f}")
print(f"Updated state: {estimator.x} +/- {to_errors(estimator.C)}")

## Optional reference: covariance and measurement models

The exercise steps above are sufficient for E1-E7. This section and the software mapping below provide details for inspecting the implementation.

The displayed F and B matrices are derivatives of the predicted state with respect to the previous state and acceleration. The code uses $Q=B\Sigma_aB^T$, with $\Sigma_a=\sigma_a^2I$: an unknown acceleration perturbation held over one interval. In particle tracking, process uncertainty can instead represent effects such as multiple scattering in detector material.

The Joseph update,
$$C_k=(I-K_kH_k)C_k^{k-1}(I-K_kH_k)^T+K_kV_kK_k^T,$$
preserves covariance structure more reliably under floating-point roundoff. The filtered residual has covariance $R_k=V_k-H_kC_kH_k^T$. The model assumes interval-held acceleration and additive, unbiased noise.

The flow model selects X/Y velocity; the simple range model measures world Z with suitable surface and sensor geometry. The onboard state also includes attitude, and its [flow and Lighthouse measurement models](https://www.bitcraze.io/documentation/repository/crazyflie-firmware/master/functional-areas/sensor-to-control/state_estimators/#measurement-models) are more detailed.


### What the live data represent

`cflib` reads the drone's log channels. The course filter runs on the laptop, using onboard position `stateEstimate.x/y/z` as its observation and world-frame acceleration as its input. The helper uses the onboard orientation to rotate body-frame specific force into world coordinates, converts g to m/s² and subtracts gravity.

Onboard position and attitude already use inertial data, so these inputs share sensor information. Live comparisons show consistency; simulation supplies independent truth for measuring error and coverage. The optional reference at the end maps all symbols to code and log channels.


## Optional: inspect a six-state recording

Set `inspection_directory` to a complete saved six-state capture to inspect body-frame specific force, onboard attitude, world-frame acceleration, position and velocity. Those onboard estimates are a separate comparison. E3-E7 use the vertical sensor format described above.


In [ ]:
inspection_directory = None  # Or select the folder containing a checked live capture.
overview = None
capture_dir = inspection_directory

if capture_dir is not None:
    overview = load_capture_overview(capture_dir)
    overview_path = next_attempt(output / "inspection") / "sensor-overview.png"
    plot_capture_overview(overview, overview_path)
    display(Image(filename=str(overview_path)))
    print("Saved:", overview_path)
else:
    print("No six-state recording selected; optional inspection skipped.")


### Reference: from Frühwirth's equations to Crazyflie data

`cflib` connects to the drone and delivers named firmware log values. The course's `PositionVelocityKalmanFilter` performs the six-state calculation on the laptop. These are two separate software layers.

| Paper quantity | Course Python code | Connection to the drone |
| --- | --- | --- |
| $k$, $\Delta t$ | `ReplayFilter.step(time_s, acceleration, position)` | Firmware log timestamps are unwrapped and converted from milliseconds to seconds; consecutive accepted sample times set `dt`. |
| $x_k=(p_x,p_y,p_z,v_x,v_y,v_z)^T$ | `estimator.x` after `update_position`; `.position` and `.velocity` select its two halves | This is the laptop estimate. Its velocity is inferred by the model and updates. |
| $m_k$ for a position update | `position`, passed to `update_position(position)` | The live helper reads `stateEstimate.x`, `.y`, `.z`, in world-frame metres. |
| $\mathbf{a}_{k-1}$, process input | `acceleration`, passed to `predict(dt, acceleration)` | `acc.x`, `.y`, `.z` supply body-frame specific force in g. The helper subtracts the chosen bias, rotates with `stateEstimate.qx`, `.qy`, `.qz`, `.qw`, multiplies by 9.80665, then subtracts world gravity. |
| $F_{k-1}$, $B_{k-1}$ | `position_velocity_model(dt)` | Model matrices calculated on the laptop. |
| $Q_{k-1}$ | `B @ (I * config.acceleration_noise_std**2) @ B.T` | Chosen uncertainty of an unknown acceleration perturbation held over the interval. |
| $H_k$ | `H` in `update_position`: `[I, 0]` | Selects the three position components for comparison with the logged position. |
| $V_k$ | `I * config.position_noise_std**2`; saved as `last_V` | Assumed position-measurement error covariance assigned to the observation used by the course filter. |
| $C_k^{k-1}$, then $C_k$ | `estimator.C` after prediction, then after update | Covariance of the six-component state-estimation error, before and after the observation. |
| $r_k^{k-1}$, $R_k^{k-1}$ | `last_predicted_residual`, `last_predicted_residual_covariance` | Observation minus the predicted measurement; R is the covariance of this predicted residual, saved before the update. |
| $K_k$, $\Delta\chi_k^2$ | `kalman_gain`, `last_chi2`; `config.chi2_gate` sets a rejection threshold | The gain uses predicted state-error covariance C and measurement-error covariance V. Chi-square uses the predicted residual and its covariance R. |

The live path is `cflib` logging → `SampleAssembler` in [live.py](live.py) → `ReplayFilter.step` in [recording.py](../kalman/recording.py) → `predict` and `update_position` in [filter.py](../kalman/filter.py). A `LogConfig` selects channel names; its `data_received_cb` receives `(timestamp, data, config)`, with readings such as `data['stateEstimate.x']`. The helper handles timing, units and the verified firmware motor stop before passing samples to the filter.

The state-estimation error covariance $C$ describes uncertainty in the estimated position and velocity. Each diagonal entry is a **state-estimation error variance**: position variances have units m² and velocity variances (m/s)². For example, $C_{11}=0.01$ m² corresponds to a position-error standard deviation of 0.1 m. Position-velocity covariance entries have units m²/s; off-diagonal entries describe correlated errors. Prediction propagates C with F and adds Q; the measurement update produces the filtered state-estimation error covariance. This is maintained by the course filter, separately from the onboard estimator's covariance. Measurement-error covariance V describes the observations, and predicted-residual covariance R combines their uncertainty with the predicted state uncertainty.

A unit quaternion is a four-component representation of orientation; the logged order is x,y,z,w. The helper uses it to rotate a body-axis vector into the world frame, whose Z axis points upward. At rest, rotation of the accelerometer's specific force gives approximately +g in world Z; subtracting gravity then gives approximately zero translational acceleration.

**Interpretation:** `stateEstimate.*` values are already estimates produced onboard, using positioning and inertial data. Here they become observations $m_k$ of a second, educational filter. The onboard position, quaternion and accelerometer channels can therefore be correlated. Agreement between the two estimates alone cannot establish accuracy or validate the independent-noise assumptions. The prepared simulation supplies known truth for that comparison.

The live helper supplies position observations and acceleration inputs. Flow velocity, range and zero-velocity updates are additional model interfaces used in the other experiments. To connect a new sensor, specify its measurement vector $m_k$, prediction $h_k(x)$, Jacobian $H_k$, covariance $V_k$, frame, units and timestamp.
